In [1]:
%pip install shap -q

import shap
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import boto3
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

BUCKET = "cancer-source-data"

# Reload model
import tarfile
s3 = boto3.client('s3')
s3.download_file(BUCKET, "models/xgboost-model/model.tar.gz", "model.tar.gz")
with tarfile.open("model.tar.gz", "r:gz") as tar:
    tar.extractall("model/")
model = xgb.Booster()
model.load_model("model/xgboost-model")

# Rebuild test set
df = pd.read_parquet(f"s3://{BUCKET}/processed/ml-ready/")
labels = df[['tumor_sample_barcode', 'cancer_type']].drop_duplicates()
labels = labels.set_index('tumor_sample_barcode')['cancer_type']
matrix = pd.crosstab(df['tumor_sample_barcode'], df['hugo_symbol'])
matrix = (matrix > 0).astype('int8')
matrix['cancer_type'] = labels
le = LabelEncoder()
matrix['label'] = le.fit_transform(matrix['cancer_type'])
gene_cols = [c for c in matrix.columns if c not in ('cancer_type', 'label')]
X = matrix[gene_cols]
y = matrix['label']
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
# shap_values shape: (95, 363, 5) — one matrix per class

class_names = ['BRCA', 'COAD', 'KIRC', 'LUAD', 'PRAD']

# Plot 1: summary beeswarm per cancer type
for i, cancer in enumerate(class_names):
    plt.figure()
    shap.summary_plot(
        shap_values[:, :, i],
        X_test,
        max_display=15,
        show=False,
        plot_type="dot"
    )
    plt.title(f"SHAP — top genes driving {cancer} prediction")
    plt.tight_layout()
    plt.savefig(f"shap_{cancer}.png", dpi=150, bbox_inches='tight')
    plt.close()
    s3.upload_file(f"shap_{cancer}.png", BUCKET, f"output/plots/shap_{cancer}.png")
    print(f"Saved shap_{cancer}.png")

# Plot 2: single bar chart — top 20 genes by mean absolute SHAP across all classes
mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))  # average over patients and classes
top20_idx = np.argsort(mean_abs_shap)[-20:][::-1]
top20_genes = X_test.columns[top20_idx]
top20_vals  = mean_abs_shap[top20_idx]

plt.figure(figsize=(8, 6))
plt.barh(top20_genes[::-1], top20_vals[::-1], color='steelblue')
plt.xlabel("Mean |SHAP value|")
plt.title("Top 20 most predictive genes (all cancer types)")
plt.tight_layout()
plt.savefig("top20_genes.png", dpi=150)
plt.close()
s3.upload_file("top20_genes.png", BUCKET, "output/plots/top20_genes.png")
print("Saved top20_genes.png")

print("\nAll SHAP plots saved to S3.")

Note: you may need to restart the kernel to use updated packages.


/Users/eesha/cancer-classifier/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/eesha/cancer-classifier/venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Saved shap_BRCA.png
Saved shap_COAD.png
Saved shap_KIRC.png
Saved shap_LUAD.png
Saved shap_PRAD.png
Saved top20_genes.png

All SHAP plots saved to S3.
